# 05 - Held-out test set: RAG vs agent

Runs the two final systems once on the held-out test questions (`golden/test_set.jsonl`):

1. RAG: company/year filter + reranker, top-5 passages, one answer call (notebook 02 setup);
2. agent v2: plan, searches, check, answer (notebook 04 setup).

Same model (Qwen3.5-9B, 4-bit), same prompt, greedy decoding. The test questions were not used to
build or tune either system, so run this notebook once, after the code is frozen.

Before running: Runtime > T4 GPU, Colab Secret `HF_TOKEN` (notebook access on). About 40 minutes.

In [ ]:
import os
import subprocess
import sys

REPO = "agentic-rag-eval"
BRANCH = "p1-agentic-rag"
if not os.path.exists(REPO):
    subprocess.run(
        ["git", "clone", "-b", BRANCH, f"https://github.com/ilaria-cesa/{REPO}.git"], check=True
    )
else:
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
%pip install -q -r requirements.txt

In [ ]:
import gc
import json
import random
import time

import numpy as np
import pandas as pd
import torch
from google.colab import userdata
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer

from src.agent.graph import AgentConfig, build_agent, regex_fallback, run_agent
from src.agent.llm import generate, load_llm
from src.agent.parsing import parse_answer
from src.agent.prompts import build_messages
from src.eval.golden import load_golden, validate
from src.eval.metrics import retrieval_metrics, score_row, summarise
from src.retrieval.embed import embed_texts, load_chunks
from src.retrieval.pipeline import Corpus, RetrievalConfig, retrieve, scoped_search
from src.retrieval.rerank import Reranker
from src.retrieval.scope import filings_of

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
HF_REPO = "IlariaCesa/sec-10k-chunks"
chunks = load_chunks(hf_hub_download(HF_REPO, "chunks_1600.jsonl", repo_type="dataset"))
embeddings = np.load(
    hf_hub_download(HF_REPO, "embeddings_bge-m3_1600.npy", repo_type="dataset")
).astype(np.float32)
assert len(chunks) == len(embeddings)
questions = load_golden("golden/test_set.jsonl")
errors, _ = validate(questions, chunks)
assert not errors, errors

corpus = Corpus(chunks, embeddings)
filings = filings_of(chunks)
names = {c["ticker"]: c["company"] for c in chunks}
embedder = SentenceTransformer("BAAI/bge-m3", device="cuda", model_kwargs={"dtype": torch.float16})
embedder.max_seq_length = 640
reranker = Reranker()


def search(query, ticker, year, k):
    vector = embed_texts(embedder, [query], progress=False)[0]
    return scoped_search(corpus, vector, query, ticker, year, k, reranker)


def score_fn(question, passages):
    return reranker.score(question, [c["embed_text"] for c in passages])


print(len(chunks), "chunks,", len(questions), "questions")

In [ ]:
def make_llm(tokenizer, model):
    def llm(messages, max_new_tokens):
        return generate(tokenizer, model, messages, max_new_tokens)["text"]

    return llm


NAME = "Qwen/Qwen3.5-9B"
CONFIG = RetrievalConfig("filter + rerank", use_filter=True, rerank=True)
K = 5
gc.collect()
torch.cuda.empty_cache()
free_gb = torch.cuda.mem_get_info()[0] / 1e9
assert free_gb > 9, "GPU memory is taken by an earlier run: Runtime > Restart session, Run all"
tokenizer, model, loader = load_llm(NAME)
print("loaded with", loader)
llm = make_llm(tokenizer, model)
agent = build_agent(
    llm,
    search,
    filings,
    names,
    AgentConfig(),
    default_scope=regex_fallback(filings),
    score_fn=score_fn,
)

rag_rows, agent_rows = [], []
for q in questions:
    # 1) plain RAG
    start = time.time()
    vec = embed_texts(embedder, [q["question"]], progress=False)[0]
    idx, _ = retrieve(corpus, q["question"], vec, CONFIG, reranker)
    retrieved_ids = [chunks[i]["chunk_id"] for i in idx]
    top = [chunks[i] for i in idx[:K]]
    provided = [c["chunk_id"] for c in top]
    out = generate(tokenizer, model, build_messages(q["question"], top))
    parsed = parse_answer(out["text"], provided)
    row = score_row(q, parsed, retrieved_ids)
    if q["expected"] == "answer":
        row |= retrieval_metrics(q["evidence_chunk_ids"], retrieved_ids)
    row |= {
        "model": NAME,
        "question": q["question"],
        "answer": parsed["answer"],
        "cited_ids": parsed["cited_ids"],
        "retrieved_top5": provided,
        "seconds": round(time.time() - start, 1),
    }
    rag_rows.append(row)

    # 2) agent v2
    start = time.time()
    out = run_agent(agent, q["question"])
    row = score_row(q, out, out["evidence_ids"])
    if q["expected"] == "answer":
        row |= retrieval_metrics(q["evidence_chunk_ids"], out["evidence_ids"])
    row |= {
        "model": NAME,
        "question": q["question"],
        "answer": out["answer"],
        "cited_ids": out["cited_ids"],
        "evidence_ids": out["evidence_ids"],
        "rounds": out["rounds"],
        "trace": out["trace"],
        "seconds": round(time.time() - start, 1),
    }
    agent_rows.append(row)
    print(
        q["id"],
        q["type"],
        "rag:",
        "ok" if rag_rows[-1]["behaviour_ok"] else "WRONG",
        "agent:",
        "ok" if row["behaviour_ok"] else "WRONG",
    )

os.makedirs("results", exist_ok=True)
for tag, rows in [("test_rag", rag_rows), ("test_agent_v2", agent_rows)]:
    with open(f"results/{tag}.jsonl", "w") as f:
        f.writelines(json.dumps(r, ensure_ascii=False) + "\n" for r in rows)

In [ ]:
pd.set_option("display.width", 200)
columns = [
    "n",
    "behaviour_ok",
    "refused",
    "invalid_citation",
    "cites_evidence",
    "numbers_match",
    "hit@5",
    "all@5",
    "all@10",
    "mrr",
    "seconds",
]
summaries = {}
for tag, rows in [("test_rag", rag_rows), ("test_agent_v2", agent_rows)]:
    summaries[tag] = summarise(rows)
    print("\n" + tag)
    print(pd.DataFrame(summaries[tag]).T.reindex(columns=columns).to_string())
with open("results/test_summary.json", "w") as f:
    json.dump(summaries, f, indent=1)

In [ ]:
import shutil

from google.colab import files

shutil.make_archive("test_results", "zip", "results")
files.download("test_results.zip")